In [42]:
import numpy as np

# Toy dataset: features are continuous
X_train = np.array([
    [5.1, 3.5],
    [4.9, 3.0],
    [6.2, 3.4],
    [5.9, 3.0],
    [7.0, 3.2],
    [6.4, 3.2]
])

y_train = np.array([0, 0, 1, 1,0, 2])  # 3 classes

# Query point
x_query = np.array([6.0, 3.0])

# Compute class priors
classes = np.unique(y_train)
priors = {c: np.mean(y_train == c) for c in classes}
priors

{np.int64(0): np.float64(0.5),
 np.int64(1): np.float64(0.3333333333333333),
 np.int64(2): np.float64(0.16666666666666666)}


$$
P(y|x) = \frac{P(x|y) P(y)}{P(x)}  \quad ; \quad P(y|x_1,x_2,...,x_n)\propto P(y) \prod_{i=1}^{n} P(x_i|y)
$$

- $P(Y)$ → Prior ( how common the class is)
- $P(X|Y)$ → Likelihood ( how likely the features are given the class.
- $P(X)$ → Evidence ( normalization constant). This is same for every class because it is compute by summing over all classes.


### Gaussian Naive Bayes for continuous features

In [43]:
import numpy as np

# Toy dataset: features are continuous
X_train = np.array([
    [5.1, 3.5],
    [4.9, 3.0],
    [6.2, 3.4],
    [5.9, 3.0],
    [7.0, 3.2],
    [6.4, 3.2]
])

y_train = np.array([0, 0, 1, 1, 2, 2])  # 3 classes

# Query point
x_query = np.array([6.0, 3.0])

# Compute class priors
classes = np.unique(y_train)
priors = {c: np.mean(y_train == c) for c in classes}
priors

{np.int64(0): np.float64(0.3333333333333333),
 np.int64(1): np.float64(0.3333333333333333),
 np.int64(2): np.float64(0.3333333333333333)}

$$
\begin{align*}
P(x_i \mid Y = C_k)&=\frac{1}{\sqrt{2\pi\sigma_{ik}^2}}\exp\!\left(-\frac{(x_i - \mu_{ik})^2}{2\sigma_{ik}^2}\right)\end{align*}
$$

In [2]:
section_data = X_train[y_train==0]
print(section_data.mean(axis=0)) # mean of the features (with respect to class=0)
print(section_data.var(axis=0)) # variance of the features ( with respect to class=1)

[5.   3.25]
[0.01   0.0625]


In [3]:
# Compute class-wise mean & variance (MLE)
mean_var = {}
for c in classes:
    X_c = X_train[y_train == c]
    mean_var[c] = (X_c.mean(axis=0), X_c.var(axis=0))

mean_var

{np.int64(0): (array([5.  , 3.25]), array([0.01  , 0.0625])),
 np.int64(1): (array([6.05, 3.2 ]), array([0.0225, 0.04  ])),
 np.int64(2): (array([6.7, 3.2]), array([0.09, 0.  ]))}

In [4]:
x_query

array([6., 3.])

In [5]:
# Gaussian likelihood
def gaussian_likelihood(x, mean, var):
    eps = 1e-8  # numerical stability
    coeff = 1 / np.sqrt(2 * np.pi * var + eps)
    exponent = np.exp(- (x - mean) ** 2 / (2 * var + eps))
    return coeff * exponent

# Compute posterior for each class
posteriors = {}
for c in classes:
    mean, var = mean_var[c]
    likelihood = np.prod(gaussian_likelihood(x_query, mean, var))
    posteriors[c] = priors[c] * likelihood

# Predicted class : class with maximum posterior 
pred_class = max(posteriors, key=posteriors.get)
pred_class, posteriors

(np.int64(1),
 {np.int64(0): np.float64(2.4825520725862847e-22),
  np.int64(1): np.float64(1.014618831272561),
  np.int64(2): np.float64(0.0)})

### Bernoulli Naive Bayes ( Binary features)

$$
\begin{align*}P(x_i \mid Y)&=p_i^{x_i}\,(1 - p_i)^{1 - x_i}\end{align*}
$$

Naive Bayes naturally supports mixed feature types - you just use a different likelihood model for each feature, and multiply them all together.  - independent likelihoods, each with its own distribution.

Use Bernoulli Naive Bayes when features represent binary presence/absence, and repeated occurrences carry no additional meaning.  

In [6]:
data = np.array([
    [1, 1, 1],
    [0, 1, 0],
    [1, 1, 0],
    [0, 0, 1]
])

In [7]:
feature_probability = data.sum(axis=0)+1 / data.shape[0]# feature wise sum  
feature_probability

array([2.25, 3.25, 2.25])

In [8]:
x_query_bin = np.array([1, 0, 1])
x_query_bin

array([1, 0, 1])

In [20]:
import numpy as np
# Toy binary dataset
X_train_bin = np.array([
    [1, 0, 1],
    [0, 1, 0],
    [1, 1, 0],
    [0, 0, 1]
])
y_train_bin = np.array([0, 1, 1, 0])

# Query point
x_query_bin = np.array([1, 0, 1])

# Step 1: Priors
classes = np.unique(y_train_bin)
priors = {c: np.mean(y_train_bin == c) for c in classes}

# Step 2: Likelihood (Bernoulli)
likelihoods = {}
for c in classes:
    print('class = ',c)
    X_c = X_train_bin[y_train_bin == c]
    # Feature-wise probability with Laplace smoothing
    feature_prob = (X_c.sum(axis=0) + 1) / (X_c.shape[0] + 2)
    print('feature probability = ',feature_prob)
    print('x_query_bin = ',x_query_bin)
    
    # Bernoulli likelihood
    print('multiplication value =',feature_prob ** x_query_bin * (1 - feature_prob) ** (1 - x_query_bin))
    likelihood = np.prod(feature_prob ** x_query_bin * (1 - feature_prob) ** (1 - x_query_bin))
    
    likelihoods[c] = priors[c] * likelihood
    print('*'*100)

class =  0
feature probability =  [0.5  0.25 0.75]
x_query_bin =  [1 0 1]
multiplication value = [0.5  0.75 0.75]
****************************************************************************************************
class =  1
feature probability =  [0.5  0.75 0.25]
x_query_bin =  [1 0 1]
multiplication value = [0.5  0.25 0.25]
****************************************************************************************************


In [23]:
# Prediction
pred_class = max(likelihoods, key=likelihoods.get)
pred_class

np.int64(0)

In [24]:
likelihoods

{np.int64(0): np.float64(0.140625), np.int64(1): np.float64(0.015625)}

### Multinomial Naive Bayes

- Class conditional probabilities:   $P(w \mid c) = [0.5,\ 0.3,\ 0.1,\ 0.05,\ 0.05]$
- Query vector (word counts): $x = [2,\ 1,\ 0,\ 0,\ 0] $
- likelihood computation : $p(x \mid c) = \prod_{i=1}^{V} P(w_i \mid c)^{x_i}$
- Substitute values: $P(x \mid c) = (0.5)^2 \cdot (0.3)^1 \cdot (0.1)^0 \cdot (0.05)^0 \cdot (0.05)^0$


In [30]:
import numpy as np

# Each row = one document (not one class!)
# Columns = counts of 5 animals: [cat, dog, cow, lion, tiger]

X_train = np.array([
    # Class 0 (e.g., "domestic animals")
    [5, 4, 0, 0, 0],
    [6, 3, 1, 0, 0],
    [4, 5, 0, 0, 0],

    # Class 1 (e.g., "farm animals")
    [0, 0, 6, 0, 0],
    [1, 0, 7, 0, 0],
    [0, 1, 5, 0, 1],

    # Class 2 (e.g., "wild animals")
    [0, 0, 0, 4, 5],
    [0, 0, 0, 5, 4],
    [0, 0, 1, 6, 6],
])

y_train = np.array([
    0, 0, 0,   # domestic
    1, 1, 1,   # farm
    2, 2, 2    # wild
])

# Query document
x_query = np.array([1, 1, 0, 0, 0])  # mostly domestic animals

In [32]:
X_train

array([[5, 4, 0, 0, 0],
       [6, 3, 1, 0, 0],
       [4, 5, 0, 0, 0],
       [0, 0, 6, 0, 0],
       [1, 0, 7, 0, 0],
       [0, 1, 5, 0, 1],
       [0, 0, 0, 4, 5],
       [0, 0, 0, 5, 4],
       [0, 0, 1, 6, 6]])

In [33]:
y_train

array([0, 0, 0, 1, 1, 1, 2, 2, 2])

In [34]:
# Step 1: Prior probability P(c)
classes = np.unique(y_train)
priors = {}

for c in classes:
    priors[c] = np.mean(y_train == c)  
    # fraction of documents belonging to class c

# Step 2: Likelihood P(word | class)
alpha = 1  # Laplace smoothing to avoid zero probabilities
likelihoods = {}

for c in classes:
    
    # all documents belonging to class c
    X_c = X_train[y_train == c]
    
    # total word counts across all docs in class c
    class_word_counts = X_c.sum(axis=0)   # shape: (num_features,)
    
    # total number of words in class c
    total_words = class_word_counts.sum()
    
    # Multinomial parameter estimate:
    # P(word_i | c) = (count_i + alpha) / (total_words + alpha * V)
    V = X_train.shape[1]
    prob_w_given_c = (class_word_counts + alpha) / (total_words + alpha * V)
    
    # Step 3: Compute log likelihood for numerical stability
    # log P(x|c) = sum_i x_i * log P(word_i|c)
    log_likelihood = np.sum(x_query * np.log(prob_w_given_c))
    
    # posterior (in log space)
    log_posterior = np.log(priors[c]) + log_likelihood
    
    likelihoods[c] = log_posterior

# Step 4: Prediction
pred_class = max(likelihoods, key=likelihoods.get)

pred_class, likelihoods

(np.int64(0),
 {np.int64(0): np.float64(-2.7540893318997526),
  np.int64(1): np.float64(-6.2285110035911835),
  np.int64(2): np.float64(-8.265650165580329)})

### Mixed Data Type : Implementation: 

In [35]:
import numpy as np

class NaiveBayes:
    def __init__(self, X, y, smoothing, feature_types):
        self.X = np.array(X)
        self.y = np.array(y)
        self.laplace = smoothing
        self.feature_types = feature_types
        self.classes = np.unique(y)
        self.class_priors = {}
        self.parameters = {}
        self._fit()

    def _fit(self):
        n_samples = len(self.y)
        # Compute class priors
        self.class_priors = {c: np.sum(self.y == c)/n_samples for c in self.classes}
        
        self.parameters = {c: {} for c in self.classes}

        for c in self.classes:
            X_c = self.X[self.y == c] # taking the respective columns 
            for idx in range(self.X.shape[1]): # each column/feature
                ftype = self.feature_types[idx]
                values = X_c[:, idx]

                if ftype == 'continuous':
                    mean = np.mean(values.astype(float))
                    std = np.std(values.astype(float)) + 1e-6  # avoid zero std
                    self.parameters[c][idx] = {'type':'continuous', 'mean':mean, 'std':std}

                elif ftype == 'level':
                    unique_vals, counts = np.unique(values, return_counts=True)
                    total = np.sum(counts)
                    probs = {v: (counts[i] + self.laplace)/(total + self.laplace*len(unique_vals))
                             for i, v in enumerate(unique_vals)}
                    self.parameters[c][idx] = {'type':'level', 'probs':probs, 'levels':unique_vals}

                elif ftype == 'count':
                    # Treat as multinomial/count: probability proportional to value
                    total = np.sum(values.astype(float))
                    self.parameters[c][idx] = {'type':'count', 'total':total, 'values':values.astype(float)}
            

    def _gaussian_likelihood(self, x, mean, std):
        exponent = -0.5 * ((x - mean)/std)**2
        return (1 / (np.sqrt(2*np.pi) * std)) * np.exp(exponent)

    def predict(self, X_test):
        X_test = np.array(X_test)
        y_pred = []
        for sample in X_test:
            log_probs = {}
            for c in self.classes:
                log_prob = np.log(self.class_priors[c])
                for idx, x_i in enumerate(sample):
                    param = self.parameters[c][idx]
                    ftype = param['type']

                    if ftype == 'continuous':
                        mean = param['mean']
                        std = param['std']
                        likelihood = self._gaussian_likelihood(float(x_i), mean, std)
                        log_prob += np.log(likelihood + 1e-9)  # avoid log(0)

                    elif ftype == 'level':
                        probs = param['probs']
                        log_prob += np.log(probs.get(x_i, self.laplace*1e-3))  # unseen level handling

                    elif ftype == 'count':
                        total = param['total']
                        prob = (float(x_i) + self.laplace) / (total + self.laplace*len(param['values']))
                        log_prob += np.log(prob + 1e-9)

                log_probs[c] = log_prob
            y_pred.append(max(log_probs, key=log_probs.get))
        return np.array(y_pred)


In [36]:
x = np.array([
    [23.1, 74, 'yes', 'small', 1],
    [18.5, 65, 'no', 'medium', 0],
    [30.2, 80, 'yes', 'large', 1],
    [25.0, 70, 'no', 'small', 0],
    [20.1, 68, 'yes', 'medium', 1],
    [22.3, 72, 'no', 'large', 0],
    [19.8, 66, 'yes', 'small', 1],
    [24.5, 75, 'no', 'medium', 0],
    [21.0, 69, 'yes', 'large', 1],
    [23.9, 73, 'no', 'medium', 0],
    [26.1, 78, 'yes', 'small', 1],
    [18.9, 64, 'no', 'large', 0],
    [29.5, 79, 'yes', 'medium', 1],
    [22.0, 71, 'no', 'small', 0],
    [20.5, 67, 'yes', 'medium', 1]
])
y = np.array([0, 2, 0, 1, 0, 1, 0, 1, 2, 1, 1, 2, 0, 1, 1])
feature_types = ['continuous','continuous','level','level','level']


In [37]:
nb = NaiveBayes(x,y,smoothing=1,feature_types=feature_types)

In [38]:
nb._fit()

In [39]:
nb.parameters

{np.int64(0): {0: {'type': 'continuous',
   'mean': np.float64(24.54),
   'std': np.float64(4.492038399666213)},
  1: {'type': 'continuous',
   'mean': np.float64(73.4),
   'std': np.float64(5.642695391866353)},
  2: {'type': 'level',
   'probs': {np.str_('yes'): np.float64(1.0)},
   'levels': array(['yes'], dtype='<U32')},
  3: {'type': 'level',
   'probs': {np.str_('large'): np.float64(0.25),
    np.str_('medium'): np.float64(0.375),
    np.str_('small'): np.float64(0.375)},
   'levels': array(['large', 'medium', 'small'], dtype='<U32')},
  4: {'type': 'level',
   'probs': {np.str_('1'): np.float64(1.0)},
   'levels': array(['1'], dtype='<U32')}},
 np.int64(1): {0: {'type': 'continuous',
   'mean': np.float64(23.471428571428568),
   'std': np.float64(1.8069038637930912)},
  1: {'type': 'continuous',
   'mean': np.float64(72.28571428571429),
   'std': np.float64(3.282608226593159)},
  2: {'type': 'level',
   'probs': {np.str_('no'): np.float64(0.6666666666666666),
    np.str_('yes'): 